# SO-app minimo: Ubuntu pelado + escritorio (una sola celda)

Sistema minimo que arranca siempre. Los programas los instalas tu despues
desde la terminal del escritorio con `apt-get`.
Abre el link `.../vnc.html` en el telefono. Toque = clic.
Probado 2026-10-13 de punta a punta (RC=0, escritorio por tunel): base 22.04.5, policy-rc.d, chequeo sin proot, sin bloque de 149 .deb.


In [ ]:
import os
import re
import shutil
import socket
import subprocess
import tarfile
import time
import urllib.request

DIR = "/kaggle/working/vmdata/proot"
ROOTFS = DIR + "/rootfs"
VNC_PORT, NOVNC_PORT = 5900, 6080
os.environ["PATH"] = ("/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:"
                      + os.environ.get("PATH", ""))
os.environ["LC_ALL"] = "C"
os.environ["LANG"] = "C"

MARK = os.path.join(ROOTFS, ".soapp-ok")


def log(m=""):
    print(m, flush=True)


def run(cmd, check=False):
    log("$ " + " ".join(cmd))
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, errors="replace")
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError("fallo rc=%d" % p.returncode)
    return p.returncode


def grab(cmd):
    r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                       text=True, errors="replace")
    return r.returncode, r.stdout


def proot(*args):
    return ["/usr/bin/proot", "-R", ROOTFS, "-b", "/dev", "-b", "/proc",
            "-b", "/sys", "-w", "/"] + list(args)


def port_open(port):
    try:
        socket.create_connection(("127.0.0.1", port), timeout=2).close()
        return True
    except OSError:
        return False


APT = ["/usr/bin/apt-get", "-o", "APT::Sandbox::User=root", "-o", "Acquire::ForceIPv4=true"]
TIMEOUT = "/usr/bin/timeout"
os.makedirs(DIR, exist_ok=True)
os.makedirs("/kaggle/working", exist_ok=True)
RUNDIR = os.path.join(DIR, "run")
TMPW = os.path.join(RUNDIR, "tmp")
LOGD = os.path.join(RUNDIR, "logs")
BINDIR = os.path.join(DIR, "bin")
for _d in (RUNDIR, TMPW, LOGD, BINDIR):
    os.makedirs(_d, exist_ok=True)
os.environ["TMPDIR"] = TMPW

log("=== [1/4] proot ===")
if not os.path.exists("/usr/bin/proot"):
    log("instalando proot...")
    run(["/usr/bin/apt-get", "update"], check=True)
    run(["/usr/bin/apt-get", "install", "-y", "proot"], check=True)
if not os.path.exists("/usr/bin/proot"):
    raise SystemExit("ERROR: no hay proot ni se pudo instalar")
log("proot OK")

log("=== [2/4] sistema minimo ===")
def _sano():
    for c in ["bin/sh", "usr/bin/dpkg", "usr/bin/apt-get", "usr/bin/tar"]:
        if not os.path.isfile(os.path.join(ROOTFS, c)):
            return False
    rc, _ = grab(proot("/bin/sh", "-c", "true"))
    return rc == 0
if _sano():
    if not os.path.isfile(MARK):
        open(MARK, "w").write("ok\n")
    log("sistema existente y funcional, reuso")
else:
    log("sistema ausente o danado, reconstruyendo base...")
    tb = os.path.join(DIR, "ubuntu-base.tar.gz")
    if not (os.path.exists(tb) and os.path.getsize(tb) > 20000000):
        log("descargando ubuntu-base (~29MB)...")
        req = urllib.request.Request(
            "https://cdimage.ubuntu.com/ubuntu-base/releases/22.04/release/ubuntu-base-22.04.5-base-amd64.tar.gz",
            headers={"User-Agent": "soapp/0.1"})
        with urllib.request.urlopen(req, timeout=60) as r, open(tb, "wb") as f:
            shutil.copyfileobj(r, f)
    else:
        log("tarball existente, reuso")
    shutil.rmtree(ROOTFS, ignore_errors=True)
    os.makedirs(ROOTFS, exist_ok=True)
    with tarfile.open(tb) as t:
        t.extractall(ROOTFS)
    _sbin = os.path.join(ROOTFS, "usr", "sbin")
    os.makedirs(_sbin, exist_ok=True)
    _pol = os.path.join(_sbin, "policy-rc.d")
    open(_pol, "w").write("#!/bin/sh\nexit 101\n")
    os.chmod(_pol, 0o755)
    log("policy-rc.d (no arrancar servicios bajo proot) OK")
    open(MARK, "w").write("ok\n")
    log("sistema minimo OK")
run(proot("/bin/sh", "-c", "dpkg --configure -a 2>&1 | tail -1 || true"))

log("=== [3/4] escritorio (solo 3 paquetes, sin recomendados) ===")
run([TIMEOUT, "300"] + proot("/bin/sh", "-c", " ".join(APT + ["update", "-qq"])))
run([TIMEOUT, "600"] + proot("/bin/sh", "-c", " ".join(APT + ["install", "-y", "--no-install-recommends", "tigervnc-standalone-server", "openbox", "xterm"]) + " || true"))
def _faltan_gui():
    # chequeo desde el host (sin proot): lexists no sigue symlinks absolutos
    # (Xvnc es un link /usr/bin/Xvnc -> /etc/alternatives/Xvnc -> Xtigervnc)
    falt = []
    for b in ["usr/bin/Xvnc", "usr/bin/openbox-session", "usr/bin/xterm"]:
        if not os.path.lexists(os.path.join(ROOTFS, b)):
            falt.append(b.split("/")[-1])
    if not os.path.isfile(os.path.join(ROOTFS, "usr/bin/Xtigervnc")):
        falt.append("Xtigervnc")
    return falt
faltan = _faltan_gui()
if faltan:
    raise SystemExit("ERROR: sin %s despues de apt limpio - pega el bloque [3/4]" % ",".join(faltan))
log("Xvnc + openbox + xterm OK")

NOVNC = os.path.join(DIR, "novnc")
if not os.path.isfile(os.path.join(NOVNC, "vnc.html")):
    log("descargando noVNC...")
    urllib.request.urlretrieve("https://github.com/novnc/noVNC/archive/refs/heads/master.tar.gz", os.path.join(DIR, "novnc.tgz"))
    shutil.rmtree(NOVNC, ignore_errors=True)
    with tarfile.open(os.path.join(DIR, "novnc.tgz")) as t:
        t.extractall(os.path.join(DIR, "novnc-x"))
    import glob as _g
    shutil.move(_g.glob(os.path.join(DIR, "novnc-x", "*"))[0], NOVNC)
else:
    log("noVNC existente, reuso")
if shutil.which("websockify") is None:
    log("instalando websockify...")
    run(["pip", "install", "-q", "websockify"], check=True)

log("=== [4/4] arrancar y tunel ===")
GXE = {"DISPLAY": "127.0.0.1:0", "HOME": "/root", "LC_ALL": "C",
       "PATH": "/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin"}
run(["/usr/bin/pkill", "-f", "Xvnc :0"])
run(["/usr/bin/pkill", "-f", "websockify.*6080"])
time.sleep(2)
xv = open(os.path.join(LOGD, "xvnc.log"), "w")
subprocess.Popen(proot("/bin/sh", "-c",
    "rm -f /tmp/.X0-lock; Xvnc :0 -geometry 1280x800 -depth 24 -listen tcp -SecurityTypes None"),
    stdout=xv, stderr=subprocess.STDOUT, start_new_session=True,
    env={**os.environ, **GXE})
ok = False
for _ in range(30):
    if port_open(VNC_PORT):
        ok = True
        break
    time.sleep(2)
if not ok:
    run(["/usr/bin/tail", "-15", os.path.join(LOGD, "xvnc.log")])
    raise SystemExit("ERROR: Xvnc no levanto")
log("Xvnc UP")
subprocess.Popen(proot("/bin/sh", "-c", "DISPLAY=127.0.0.1:0 openbox-session"),
                 stdout=open(os.path.join(LOGD, "openbox.log"), "w"), stderr=subprocess.STDOUT,
                 start_new_session=True, env={**os.environ, **GXE})
time.sleep(2)
subprocess.Popen(proot("/bin/sh", "-c", "DISPLAY=127.0.0.1:0 xterm -geometry 100x30"),
                 stdout=open(os.path.join(LOGD, "xterm.log"), "w"), stderr=subprocess.STDOUT,
                 start_new_session=True, env={**os.environ, **GXE})
log("openbox + terminal lanzados")
ws = open(os.path.join(LOGD, "websockify.log"), "w")
subprocess.Popen(["websockify", "--web", NOVNC, str(NOVNC_PORT), "127.0.0.1:%d" % VNC_PORT],
                 stdout=ws, stderr=subprocess.STDOUT, start_new_session=True)
time.sleep(3)
if not port_open(NOVNC_PORT):
    run(["/usr/bin/tail", "-15", os.path.join(LOGD, "websockify.log")])
    raise SystemExit("ERROR: websockify no levanto")
log("noVNC UP")
CFD = os.path.join(BINDIR, "cloudflared")
if not os.access(CFD, os.X_OK):
    log("descargando cloudflared...")
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CFD)
    os.chmod(CFD, 0o755)
run(["/usr/bin/pkill", "-f", "cloudflared tunnel --url http://127.0.0.1:%d" % NOVNC_PORT])
time.sleep(2)
tlog = open(os.path.join(LOGD, "tunnel-vnc.log"), "w")
subprocess.Popen([CFD, "tunnel", "--url", "http://127.0.0.1:%d" % NOVNC_PORT],
                 stdout=tlog, stderr=subprocess.STDOUT, start_new_session=True)
URL = ""
for _ in range(45):
    with open(os.path.join(LOGD, "tunnel-vnc.log"), errors="replace") as f:
        m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", f.read())
    if m:
        URL = m.group(0)
        break
    time.sleep(2)
if not URL:
    run(["/usr/bin/tail", "-15", os.path.join(LOGD, "tunnel-vnc.log")])
    raise SystemExit("ERROR: sin URL de tunel")
log("ESCRITORIO: " + URL + "/vnc.html")
log("Adentro abre la terminal y instala tus programas con apt-get.")
